## Setup and Data Preparation

In [2]:
# Import required libraries
import random
import math
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch.nn as nn
import torch
import networkx as nx
import numpy as np
import os
import warnings
os.environ["KMP_WARNINGS"] = "0"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
warnings.filterwarnings("ignore", category=UserWarning)


# Set device and random seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Graph Convolutional Network (GCN) Implementation Layer & Model Definition (provided)
def sparse_mm(A, X):
  return A @ X


class GCNLayer(nn.Module):
  def __init__(self, in_dim, out_dim, bias=True):
    super().__init__()
    self.lin = nn.Linear(in_dim, out_dim, bias=bias)

  def forward(self, A_hat, X):
    return self.lin(sparse_mm(A_hat, X))


class GCN(nn.Module):
  def __init__(self, in_dim, hid_dim, out_dim, dropout=0.2):
    super().__init__()
    self.g1 = GCNLayer(in_dim, hid_dim)
    self.g2 = GCNLayer(hid_dim, out_dim)
    self.dropout = dropout

  def forward(self, A_hat, X):
    H = F.relu(self.g1(A_hat, X))
    H = F.dropout(H, p=self.dropout, training=self.training)
    Z = self.g2(A_hat, H)
    return Z


print("GCN model defined successfully!")

In [ ]:
# Helper functions for synthetic data generation
def to_norm_adj(G):
  """Convert NetworkX graph to normalized adjacency matrix"""
  n = G.number_of_nodes()
  idx_map = {n_id: i for i, n_id in enumerate(G.nodes())}
  rows, cols = [], []
  for u, v in G.edges():
    ui, vi = idx_map[u], idx_map[v]
    rows += [ui, vi]
    cols += [vi, ui]
  indices = torch.tensor([rows, cols], dtype=torch.long)
  values = torch.ones(indices.shape[1], dtype=torch.float32)
  A = torch.sparse_coo_tensor(indices, values, (n, n)).to_dense()
  deg = A.sum(1).clamp(min=1.0)
  D_inv_sqrt = torch.pow(deg, -0.5)
  D = torch.diag(D_inv_sqrt)
  A_hat = D @ A @ D
  return A_hat.to(device)


def make_sbm(n_per_comm=100, p_in=0.08, p_out=0.01, num_comms=3, feat_dim=16, noise=0.5):
  """Create Stochastic Block Model graph with features"""
  sizes = [n_per_comm] * num_comms
  probs = [[p_in if i == j else p_out for j in range(num_comms)] for i in range(num_comms)]
  G = nx.stochastic_block_model(sizes, probs, seed=seed)
  labels = []
  for i, size in enumerate(sizes):
    labels += [i] * size
  y = torch.tensor(labels, dtype=torch.long, device=device)
  centroids = torch.eye(num_comms, feat_dim, device=device)[:num_comms]
  X = torch.zeros((sum(sizes), feat_dim), device=device)
  start = 0
  for c, size in enumerate(sizes):
    X[start:start + size] = centroids[c] + noise * torch.randn(size, feat_dim, device=device)
    start += size
  A = to_norm_adj(G)
  return A, X, y, G


def split_idx(n, val_ratio=0.1, test_ratio=0.2):
  """Split indices into train/val/test sets"""
  idx = np.arange(n)
  idx_train, idx_tmp = train_test_split(idx, test_size=val_ratio + test_ratio, random_state=seed)
  rel_test = test_ratio / (val_ratio + test_ratio)
  idx_val, idx_test = train_test_split(idx_tmp, test_size=rel_test, random_state=seed)
  return torch.tensor(idx_train), torch.tensor(idx_val), torch.tensor(idx_test)


print("Helper functions loaded!")

In [ ]:
# Generate synthetic graph data
A, X, y, G = make_sbm(n_per_comm=120, p_in=0.08, p_out=0.01, num_comms=3, feat_dim=16, noise=0.6)
n, in_dim = X.shape
num_classes = int(y.max().item() + 1)

print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Feature matrix shape: {X.shape}")
print(f"Number of communities: {num_classes}")

---
## Node Classification - Influencer Detection

**Task:** Implement a GCN model to identify influential nodes in the social network.

**Instructions:**
1. Complete the code to create influencer labels based on centrality metrics
2. Train the GCN model for binary classification
3. Answer the questions based on your results

In [ ]:
# Create influencer labels
# Fill in the blanks to compute centrality metrics and create labels

# Compute centrality metrics
deg_cent = np.array(list(nx.degree_centrality(G).values()))
eig_cent = np.array(list(nx.eigenvector_centrality_numpy(G).values()))
# BLANK 1: Fill in the centrality type
bet_cent = np.array(list(nx.___centrality(G).values()))

# Combined influence score (normalize each metric)
inf_score = (deg_cent / deg_cent.max() +
             eig_cent / eig_cent.max() / 2.0 +
             # BLANK 2: Fill in the normalization factor
             bet_cent / bet_cent.max()) / ___

# Label top 10% as influencers
# BLANK 3: Fill in the percentile for top 10%
threshold = np.percentile(inf_score, ___)
y_influencer = torch.tensor((inf_score >= threshold).astype(int), dtype=torch.long, device=device)

print(f"Influencers: {y_influencer.sum().item()} / {len(y_influencer)} "
      f"({100*y_influencer.float().mean().item():.1f}%)")

In [ ]:
# Train the GCN model
# Fill in the blanks to complete the training loop

# BLANK 4: Number of output classes for binary classification
model_inf = GCN(X.shape[1], 64, ___).to(device)
opt = torch.optim.Adam(model_inf.parameters(), lr=0.01, weight_decay=5e-4)

idx = np.arange(G.number_of_nodes())
idx_train, idx_test = train_test_split(idx, test_size=0.3, stratify=y_influencer.cpu(), random_state=42)
idx_train = torch.tensor(idx_train, device=device)
idx_test = torch.tensor(idx_test, device=device)

best_acc, best_state = 0, None
for epoch in range(150):
  model_inf.train()
  out = model_inf(A, X)
  # BLANK 5: Fill in the loss function
  loss = F.___(out[idx_train], y_influencer[idx_train])

  opt.zero_grad()
  loss.backward()
  opt.step()

  model_inf.eval()
  with torch.no_grad():
    # BLANK 6: Fill in the dimension for argmax
    preds = out[idx_test].argmax(___)
    acc = (preds == y_influencer[idx_test]).float().mean().item()

  if acc > best_acc:
    best_acc = acc
    best_state = {k: v.clone() for k, v in model_inf.state_dict().items()}

  if epoch % 30 == 0:
    print(f"[Epoch {epoch}] loss={loss.item():.4f} acc={acc:.3f}")

model_inf.load_state_dict(best_state)
print(f"\nInfluencer Classification Test Accuracy: {best_acc:.3f}")

In [ ]:
# Visualize influencer predictions
model_inf.eval()
with torch.no_grad():
  logits = model_inf(A, X)
  probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(7, 7))
nodes = nx.draw_networkx_nodes(G, pos, node_color=probs, cmap=cm.get_cmap("YlOrRd"), node_size=50)
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5)
plt.colorbar(nodes, label="Predicted Influencer Probability")
plt.title("Node Classification - Influencers")
plt.axis('off')
plt.show()